# Applied Machine Learning - Linear regression and mean square error
## Code from session 3

Dataset: Rent for apartments and houses in Manhattan.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zohaib30/streeteasy-dataset")

print("Path to dataset files:", path)

## Looking at the data: What features do we have?

In [ ]:
import pandas as pd
import numpy as np
import csv

csv_path = path+"/manhattan.csv"

df = pd.read_csv(csv_path)
print(df.head())
df.info()

## Check correlations

In [ ]:
df = df.select_dtypes(include="number")
import matplotlib.pyplot as plt

METHOD = "spearman"  # "pearson" or "spearman"; used for the plotted heatmap
 
def plot_correlation_matrix(corr, title):
    fig, ax = plt.subplots(figsize=(0.6 * len(corr.columns) + 3, 0.6 * len(corr.columns) + 2))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
 
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr.columns)))
    ax.set_yticklabels(corr.columns)
 
    for i in range(len(corr.columns)):
        for j in range(len(corr.columns)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
 
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="correlation")
    fig.tight_layout()
 
 
def main():
 
    pearson_corr = df.corr(method="pearson")
    spearman_corr = df.corr(method="spearman")
 
    print("\nPearson correlation matrix:")
    print(pearson_corr.round(3))
    print("\nSpearman correlation matrix:")
    print(spearman_corr.round(3))
 
    corr = pearson_corr if METHOD == "pearson" else spearman_corr
    plot_correlation_matrix(corr, f"{METHOD.capitalize()} correlation matrix")
 
 
if __name__ == "__main__":
    main()


In [ ]:
import matplotlib.pyplot as plt

# looking at the relevant set of variables to see if there is a meaningful linear pattern

df_correlated = df[['rent', 'bedrooms', 'bathrooms', 'size_sqft']]

pd.plotting.scatter_matrix(df_correlated, alpha=0.8, figsize=(10, 10), diagonal='hist')
plt.suptitle("Correllations with rent. Diagonal: Histogram", fontsize=16, y=0.95)
plt.show()

## Linear regression 
### for a one-dimensional feature and one-dimensional target, all written out

Model:      y_hat = w * x + b\
Loss:       Mean Squared Error, J = (1/n) * sum((y_hat - y)^2)\
Optimizer:  batch gradient descent on w and b

In [ ]:
LR = 0.05
EPOCHS = 50

x_label = "size_sqft"
y_label = "rent"

x = df[x_label].to_numpy(dtype=float)
y = df[y_label].to_numpy(dtype=float)


def mse(w1, b, x1, y):
    y_pred = w1 * x1 + b
    return np.mean((y_pred - y) ** 2)


def fit_linear_regression(x, y, lr=LR, epochs=EPOCHS):
    n = len(x)
    w, b = 0.0, 0.0
    w_history = [w]
    b_history = [b]
    loss_history = [mse(w, b, x, y)]

    for epoch in range(epochs):
        y_pred = w * x + b
        error = y_pred - y

        dw = (2.0 / n) * np.sum(x * error)
        db = (2.0 / n) * np.sum(error)

        w -= lr * dw
        b -= lr * db

        loss = mse(w, b, x, y)
        w_history.append(w)
        b_history.append(b)
        loss_history.append(loss)

        if epoch % max(1, epochs // 10) == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:5d}  loss {loss:.6f}  w {w:.6f}  b {b:.6f}")

    return w, b, np.array(w_history), np.array(b_history), np.array(loss_history)


def plot_fit(x, y, w, b, loss_history):
    order = np.argsort(x)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

    ax1.scatter(x, y, alpha=0.6, label="data")
    ax1.plot(x[order], w * x[order] + b, color="red", label="fitted line")
    ax1.set_xlabel(x_label)
    ax1.set_ylabel(y_label)
    ax1.set_title("Linear regression fit")
    ax1.legend()

    ax2.plot(loss_history)
    ax2.set_xlabel("epoch")
    ax2.set_ylabel("MSE loss")
    ax2.set_title("Training loss")

    fig.tight_layout()


def plot_gradient_descent(x_scaled, y, w_history, b_history, loss_history):
    # Grid of (w, b) values bounded by the range the descent path covers.
    w_margin = 0.3 * (w_history.max() - w_history.min() + 1e-6)
    b_margin = 0.3 * (b_history.max() - b_history.min() + 1e-6)
    w_range = np.linspace(w_history.min() - w_margin, w_history.max() + w_margin, 200)
    b_range = np.linspace(b_history.min() - b_margin, b_history.max() + b_margin, 200)
    W, B = np.meshgrid(w_range, b_range)

    Z = np.zeros_like(W)
    for i in range(W.shape[0]):
        for j in range(W.shape[1]):
            Z[i, j] = mse(W[i, j], B[i, j], x_scaled, y)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    contour = ax1.contour(W, B, Z, levels=30, cmap="viridis")
    ax1.clabel(contour, inline=True, fontsize=7)
    ax1.plot(w_history, b_history, color="red", linewidth=1.2, marker="o",
              markersize=2, label="gradient descent path")
    ax1.scatter([w_history[0]], [b_history[0]], color="black", zorder=5, label="start")
    ax1.scatter([w_history[-1]], [b_history[-1]], color="lime", zorder=5, label="end")
    ax1.set_xlabel("w (scaled x)")
    ax1.set_ylabel("b")
    ax1.set_title("MSE loss surface and gradient descent path")
    ax1.legend()

    fig.tight_layout()

def main():
    print(f"Loaded {len(x)} points from {csv_path}")

    # Standardise x for numerical stability, then convert w, b back
    # to the original scale for reporting.
    x_mean, x_std = x.mean(), x.std()
    x_scaled = (x - x_mean) / x_std

    w_s, b_s, w_history, b_history, loss_history = fit_linear_regression(x_scaled, y)

    w = w_s / x_std
    b = b_s - w_s * x_mean / x_std

    final_mse = np.mean((w * x + b - y) ** 2)
    print(f"\nFinal model: y_hat = {w:.6f} * x + {b:.6f}")
    print(f"Final MSE (original scale): {final_mse:.6f}")

    plot_fit(x, y, w, b, loss_history)
    plot_gradient_descent(x_scaled, y, w_history, b_history, loss_history)


if __name__ == "__main__":
    main()

### ... now two features

Model:      y_hat = w1*x1 + w2*x2 + b\
Loss:       Mean Squared Error, J = (1/n) * sum((y_hat - y)^2)\
Optimizer:  batch gradient descent on w and b

In [ ]:
LR = 0.05
EPOCHS = 50

x1_label = "size_sqft"
x2_label = "bedrooms"
y_label = "rent"

x1 = df[x1_label].to_numpy(dtype=float)
x2 = df[x2_label].to_numpy(dtype=float)
y = df[y_label].to_numpy(dtype=float)


def mse(w1, w2, b, x1, x2, y):
    y_pred = w1 * x1 + w2 * x2 + b
    return np.mean((y_pred - y) ** 2)


def fit_linear_regression(x1, x2, y, lr=LR, epochs=EPOCHS):
    n = len(x1)
    w1, w2, b = 0.0, 0.0, 0.0
    w1_history = [w1]
    w2_history = [w2]
    b_history = [b]
    loss_history = [mse(w1, w2, b, x1, x2, y)]
 
    for epoch in range(epochs):
        y_pred = w1 * x1 + w2 * x2 + b
        error = y_pred - y
 
        dw1 = (2.0 / n) * np.sum(x1 * error)
        dw2 = (2.0 / n) * np.sum(x2 * error)
        db = (2.0 / n) * np.sum(error)
 
        w1 -= lr * dw1
        w2 -= lr * dw2
        b -= lr * db
 
        loss = mse(w1, w2, b, x1, x2, y)
        w1_history.append(w1)
        w2_history.append(w2)
        b_history.append(b)
        loss_history.append(loss)
 
        if epoch % max(1, epochs // 10) == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:5d}  loss {loss:.6f}  w1 {w1:.6f}  w2 {w2:.6f}  b {b:.6f}")
 
    return (
        w1, w2, b,
        np.array(w1_history), np.array(w2_history), np.array(b_history),
        np.array(loss_history),
    )


def plot_fit(x1, x2, y, w1, w2, b, loss_history):
    fig = plt.figure(figsize=(12, 5))
 
    ax1 = fig.add_subplot(1, 2, 1, projection="3d")
    ax1.scatter(x1, x2, y, alpha=0.6, label="data")
 
    x1_grid, x2_grid = np.meshgrid(
        np.linspace(x1.min(), x1.max(), 20),
        np.linspace(x2.min(), x2.max(), 20),
    )
    y_grid = w1 * x1_grid + w2 * x2_grid + b
    ax1.plot_surface(x1_grid, x2_grid, y_grid, alpha=0.3, color="red")
 
    ax1.set_xlabel(x1_label)
    ax1.set_ylabel(x2_label)
    ax1.set_zlabel(y_label)
    ax1.set_title("Linear regression fit")
 
    ax2 = fig.add_subplot(1, 2, 2)
    ax2.plot(loss_history)
    ax2.set_xlabel("epoch")
    ax2.set_ylabel("MSE loss")
    ax2.set_title("Training loss")


def plot_gradient_descent(x1, x2, y, w1_history, w2_history, b_final):
    """Contour plot of the MSE loss surface in (w1, w2), with b held
    fixed at its final fitted value, and the descent path (projected
    onto the w1-w2 plane) overlaid. A full 3-parameter surface (w1,
    w2, b) cannot be drawn directly, so this is a 2D slice through it."""
    w1_margin = 0.3 * (w1_history.max() - w1_history.min() + 1e-6)
    w2_margin = 0.3 * (w2_history.max() - w2_history.min() + 1e-6)
    w1_range = np.linspace(w1_history.min() - w1_margin, w1_history.max() + w1_margin, 200)
    w2_range = np.linspace(w2_history.min() - w2_margin, w2_history.max() + w2_margin, 200)
    W1, W2 = np.meshgrid(w1_range, w2_range)
 
    Z = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            Z[i, j] = mse(W1[i, j], W2[i, j], b_final, x1, x2, y)
 
    plt.figure(figsize=(6.5, 5.5))
    contour = plt.contour(W1, W2, Z, levels=30, cmap="viridis")
    plt.clabel(contour, inline=True, fontsize=7)
    plt.plot(w1_history, w2_history, color="red", linewidth=1.0,
              marker="o", markersize=2, label="gradient descent path")
    plt.scatter([w1_history[0]], [w2_history[0]], color="black", zorder=5, label="start")
    plt.scatter([w1_history[-1]], [w2_history[-1]], color="lime", zorder=5, label="end")
    plt.xlabel("w1 (scaled x1)")
    plt.ylabel("w2 (scaled x2)")
    plt.title("MSE loss surface (b fixed at final value) and descent path")
    plt.legend()
    plt.tight_layout()


def main():
 
    # Standardise both features for numerical stability, then convert
    # the fitted weights and bias back to the original scale.
    x1_mean, x1_std = x1.mean(), x1.std()
    x2_mean, x2_std = x2.mean(), x2.std()
    x1_scaled = (x1 - x1_mean) / x1_std
    x2_scaled = (x2 - x2_mean) / x2_std
 
    (w1_s, w2_s, b_s,
     w1_history, w2_history, b_history, loss_history) = fit_linear_regression(
        x1_scaled, x2_scaled, y
    )
 
    # y = w1_s*(x1-x1m)/x1s + w2_s*(x2-x2m)/x2s + b_s
    #   = (w1_s/x1s)*x1 + (w2_s/x2s)*x2 + (b_s - w1_s*x1m/x1s - w2_s*x2m/x2s)
    w1 = w1_s / x1_std
    w2 = w2_s / x2_std
    b = b_s - w1_s * x1_mean / x1_std - w2_s * x2_mean / x2_std
 
    final_mse = np.mean((w1 * x1 + w2 * x2 + b - y) ** 2)
    print(f"\nFinal model: y_hat = {w1:.6f} * {x1_label} + {w2:.6f} * {x2_label} + {b:.6f}")
    print(f"Final MSE (original scale): {final_mse:.6f}")
 
    plot_fit(x1, x2, y, w1, w2, b, loss_history)
    plot_gradient_descent(x1_scaled, x2_scaled, y, w1_history, w2_history, b_s)
 
 
if __name__ == "__main__":
    main()

## ... now we adjust the learning rate
We lower LR whenever we overshoot, i.e. the loss function increases in the step instead of decreasing.

In [ ]:
LR_INIT = 0.1
DECAY_FACTOR = 0.5   # multiply learning rate by this whenever a step increases the loss
EPOCHS = 50
BATCH_SIZE = 1        # 1 = pure SGD; larger values give mini-batch SGD
SEED = 0


x1_label = "size_sqft"
x2_label = "bedrooms"
y_label = "rent"

x1 = df[x1_label].to_numpy(dtype=float)
y = df[y_label].to_numpy(dtype=float)


def mse(w1, w2, b, x1, x2, y):
    y_pred = w1 * x1 + w2 * x2 + b
    return np.mean((y_pred - y) ** 2)

def fit_linear_regression(x1, x2, y, lr=LR, epochs=EPOCHS):
    n = len(x1)
    w1, w2, b = 0.0, 0.0, 0.0
    w1_history = [w1]
    w2_history = [w2]
    b_history = [b]
    loss_history = [mse(w1, w2, b, x1, x2, y)]
 
    for epoch in range(epochs):
        y_pred = w1 * x1 + w2 * x2 + b
        error = y_pred - y
 
        dw1 = (2.0 / n) * np.sum(x1 * error)
        dw2 = (2.0 / n) * np.sum(x2 * error)
        db = (2.0 / n) * np.sum(error)
 
        w1 -= lr * dw1
        w2 -= lr * dw2
        b -= lr * db
 
        loss = mse(w1, w2, b, x1, x2, y)
        w1_history.append(w1)
        w2_history.append(w2)
        b_history.append(b)
        loss_history.append(loss)
 
        if epoch % max(1, epochs // 10) == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:5d}  loss {loss:.6f}  w1 {w1:.6f}  w2 {w2:.6f}  b {b:.6f}")
 
    return (
        w1, w2, b,
        np.array(w1_history), np.array(w2_history), np.array(b_history),
        np.array(loss_history),
    )
 
def plot_fit(x1, x2, y, w1, w2, b, loss_history):
    fig = plt.figure(figsize=(12, 5))
 
    ax1 = fig.add_subplot(1, 2, 1, projection="3d")
    ax1.scatter(x1, x2, y, alpha=0.6, label="data")
 
    x1_grid, x2_grid = np.meshgrid(
        np.linspace(x1.min(), x1.max(), 20),
        np.linspace(x2.min(), x2.max(), 20),
    )
    y_grid = w1 * x1_grid + w2 * x2_grid + b
    ax1.plot_surface(x1_grid, x2_grid, y_grid, alpha=0.3, color="red")
 
    ax1.set_xlabel(x1_label)
    ax1.set_ylabel(x2_label)
    ax1.set_zlabel(y_label)
    ax1.set_title("Linear regression fit")
 
    ax2 = fig.add_subplot(1, 2, 2)
    ax2.plot(loss_history)
    ax2.set_xlabel("epoch")
    ax2.set_ylabel("MSE loss")
    ax2.set_title("Training loss")
 
    fig.tight_layout() 

def plot_gradient_descent(x1, x2, y, w1_history, w2_history, b_final):
    """Contour plot of the MSE loss surface in (w1, w2), with b held
    fixed at its final fitted value, and the descent path (projected
    onto the w1-w2 plane) overlaid. A full 3-parameter surface (w1,
    w2, b) cannot be drawn directly, so this is a 2D slice through it."""
    w1_margin = 0.3 * (w1_history.max() - w1_history.min() + 1e-6)
    w2_margin = 0.3 * (w2_history.max() - w2_history.min() + 1e-6)
    w1_range = np.linspace(w1_history.min() - w1_margin, w1_history.max() + w1_margin, 200)
    w2_range = np.linspace(w2_history.min() - w2_margin, w2_history.max() + w2_margin, 200)
    W1, W2 = np.meshgrid(w1_range, w2_range)
 
    Z = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            Z[i, j] = mse(W1[i, j], W2[i, j], b_final, x1, x2, y)
 
    plt.figure(figsize=(6.5, 5.5))
    contour = plt.contour(W1, W2, Z, levels=30, cmap="viridis")
    plt.clabel(contour, inline=True, fontsize=7)
    plt.plot(w1_history, w2_history, color="red", linewidth=1.0,
              marker="o", markersize=2, label="gradient descent path")
    plt.scatter([w1_history[0]], [w2_history[0]], color="black", zorder=5, label="start")
    plt.scatter([w1_history[-1]], [w2_history[-1]], color="lime", zorder=5, label="end")
    plt.xlabel("w1 (scaled x1)")
    plt.ylabel("w2 (scaled x2)")
    plt.title("MSE loss surface (b fixed at final value) and descent path")
    plt.legend()
    plt.tight_layout()

def main(): 
    # Standardise both features for numerical stability, then convert
    # the fitted weights and bias back to the original scale.
    x1_mean, x1_std = x1.mean(), x1.std()
    x2_mean, x2_std = x2.mean(), x2.std()
    x1_scaled = (x1 - x1_mean) / x1_std
    x2_scaled = (x2 - x2_mean) / x2_std
 
    (w1_s, w2_s, b_s,
     w1_history, w2_history, b_history, loss_history) = fit_linear_regression(
        x1_scaled, x2_scaled, y
    )
 
    # y = w1_s*(x1-x1m)/x1s + w2_s*(x2-x2m)/x2s + b_s
    #   = (w1_s/x1s)*x1 + (w2_s/x2s)*x2 + (b_s - w1_s*x1m/x1s - w2_s*x2m/x2s)
    w1 = w1_s / x1_std
    w2 = w2_s / x2_std
    b = b_s - w1_s * x1_mean / x1_std - w2_s * x2_mean / x2_std
 
    final_mse = np.mean((w1 * x1 + w2 * x2 + b - y) ** 2)
    print(f"\nFinal model: y_hat = {w1:.6f} * {x1_label} + {w2:.6f} * {x2_label} + {b:.6f}")
    print(f"Final MSE (original scale): {final_mse:.6f}")
 
    plot_fit(x1, x2, y, w1, w2, b, loss_history)
    plot_gradient_descent(x1_scaled, x2_scaled, y, w1_history, w2_history, b_s)
 
 
if __name__ == "__main__":
    main()


## ... now the linear regression with the skslearn library and standard settings.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

x1_label = "size_sqft"
x2_label = "bedrooms"
y_label = "rent"
feature_cols = [x1_label, x2_label]

# 1. Define feature matrix (X) and target vector (y)
X=df[feature_cols]
y = df[y_label]

# 2. Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Instantiate and fit the model
model = LinearRegression()
model.fit(X_train, y_train)

# 4. Make predictions on the test set
y_pred = model.predict(X_test)

# 5. Model Inspection & Evaluation Metrics
print("--- Model Parameters ---")
for col, coef in zip(feature_cols, model.coef_):
    print(f"Coefficient ({col}): {coef:.4f}")
print(f"Intercept: {model.intercept_:.4f}\n")

print("--- Test Performance ---")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(
    f"Root Mean Squared Error (RMSE): {mean_squared_error(y_test, y_pred, squared=False):.4f}"
)